## **Langchain을 표현하는 언어, LCEL**

**[LCEL로 기본 체인 구성하기]**

In [1]:
#필수 라이브러리 설치
!pip install --upgrade --quiet langchain openai langchain-core langchain-openai


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

#프롬프트 템플릿 설정
prompt = ChatPromptTemplate.from_template("tell me a short joke about {topic}")

#LLM 호출
model = ChatOpenAI(model="gpt-4o-mini")

#출력 파서 설정
output_parser = StrOutputParser()

#LCEL로 프롬프트템플릿-LLM-출력 파서 연결하기
chain = prompt | model | output_parser

#invoke함수로 chain 실행하기
chain.invoke({"topic": "ice cream"})

/Users/jinhohyeon/Desktop/dev/Learned/llm/udemy-genairag-langchain-chatbot/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'What did the ice cream cone say to the ice cream sundae? \n\n"I think you\'re sweet, but I can\'t handle your toppings!"'

**[Streaming 기능 추가를 더욱 쉽게, stream()]**

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

#Chain 선언
model = ChatOpenAI(model="gpt-4o-mini")
prompt = ChatPromptTemplate.from_template("tell me a joke about {topic}")
chain = prompt | model

#Chain의 stream()함수를 통해 스트리밍 기능 추가
for s in chain.stream({"topic": "bears"}):
    print(s.content, end="", flush=True)

Why do bears have hairy coats?

Because they look silly in sweaters!

**[한꺼번에 여러 개 API 요청하고 답변 받기, batch()]**

- 5개 문장 번역 batch 수행

In [4]:
%%time
model = ChatOpenAI(model="gpt-4o-mini")
prompt = ChatPromptTemplate.from_template("다음 한글 문장을 프랑스어로 번역해줘 {sentence}")
chain = prompt | model

chain.batch([
    {"sentence": "그녀는 매일 아침 책을 읽습니다."},
    {"sentence": "오늘 날씨가 참 좋네요."},
    {"sentence": "저녁에 친구들과 영화를 볼 거예요."},
    {"sentence": "그 학생은 매우 성실하게 공부합니다."},
    {"sentence": "커피 한 잔이 지금 딱 필요해요."}

])

CPU times: user 26.5 ms, sys: 6.45 ms, total: 33 ms
Wall time: 1.56 s


[AIMessage(content='그녀는 매일 아침 책을 읽습니다.라는 문장을 프랑스어로 번역하면 다음과 같습니다.\n\n**Elle lit un livre tous les matins.**', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 33, 'total_tokens': 71, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_791df6ace0', 'id': 'chatcmpl-DgSnDZln39IZTfHvYnaLm6C1LnHLn', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e356d-76e2-7da3-8a5e-91a7a40fe44e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 33, 'output_tokens': 38, 'total_tokens': 71, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 AIMessage(content="오늘 날씨가 참 좋네요.  \n=> Au

- 1개 문장 번역 invoke 수행

In [5]:
%%time
prompt = ChatPromptTemplate.from_template("다음 한글 문장을 프랑스어로 번역해줘 {sentence}")
chain = prompt | model

chain.invoke({"sentence": "그녀는 매일 아침 책을 읽습니다."})

CPU times: user 5.34 ms, sys: 1.74 ms, total: 7.08 ms
Wall time: 1.33 s


AIMessage(content='그녀는 매일 아침 책을 읽습니다.는 프랑스어로 "Elle lit un livre chaque matin."입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 33, 'total_tokens': 61, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_791df6ace0', 'id': 'chatcmpl-DgSnJNSlHmawLPF2arRRKDDNYsQNq', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e356d-8ef2-7c92-9cce-11a6cb9edb17-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 33, 'output_tokens': 28, 'total_tokens': 61, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## **RunnablePassthrough, RunnableLambda, RunnableParallel**

**[RunnablePassthrough]**

**RunnablePassthrough는 가장 단순한 Runnable 객체로, 들어온 입력을 그대로 전달합니다.**

In [6]:
from langchain_core.runnables import RunnablePassthrough

RunnablePassthrough().invoke("안녕하세요")

'안녕하세요'

In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("다음 한글 문장을 프랑스어로 번역해줘 {sentence} \n French sentence: (print from here)")
model = ChatOpenAI(model="gpt-4o-mini")
output_parser = StrOutputParser()

runnable_chain = {"sentence": RunnablePassthrough()} | prompt | model | output_parser

runnable_chain.invoke({"sentence": "그녀는 매일 아침 책을 읽습니다."})

'Elle lit un livre chaque matin.'

RunnablePassthrough는 assin 함수를 통해 새로운 변수에 계산된 값을 입력할 수 있습니다.

In [8]:
(RunnablePassthrough.assign(mult=lambda x: x["num"]*3)).invoke({"num":3})

{'num': 3, 'mult': 9}

In [9]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

runnable = RunnableParallel(
    extra=RunnablePassthrough.assign(mult=lambda x: x["num"] * 3),
    modified=lambda x: x["num"] + 1,
)

runnable.invoke({"num": 1})

{'extra': {'num': 1, 'mult': 3}, 'modified': 2}

**[RunnableLambda]**

**RunnableLambda는 임의의 함수를 Chain에 결합할 수 있게 Runnable 객체로 변환합니다..**

In [10]:
def add_smile(x):
    return x + ":)"

In [11]:
from langchain_core.runnables import RunnableLambda

add_smile = RunnableLambda(add_smile)

In [15]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

prompt_str = "{topic}의 역사에 대해 세문장으로 설명해주세요."
prompt = ChatPromptTemplate.from_template(prompt_str)

model = ChatOpenAI(model_name = 'gpt-4o-mini')

output_parser = StrOutputParser()

chain = prompt | model | output_parser

In [16]:
from langchain_core.runnables import RunnableLambda

def add_thank(x):
    return x + " 들어주셔서 감사합니다 :)"

add_thank = RunnableLambda(add_thank)

In [17]:
chain = prompt | model | output_parser | add_thank
chain.invoke("반도체")

'반도체의 역사는 20세기 초에 시작되어, 1947년 벨 연구소에서 최초의 트랜지스터가 발명됨으로써 혁신이 시작되었습니다. 이후 1960년대에는 집적회로(IC)가 개발되어 전자기기의 소형화와 성능 향상을 이끌었습니다. 21세기 들어서는 반도체가 인공지능, 모바일, IoT 등 다양한 분야에서 핵심 기술로 자리 잡으며 산업 전반에 큰 영향을 미치고 있습니다. 들어주셔서 감사합니다 :)'

In [18]:
!pip install -q grandalf


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [19]:
chain.get_graph().print_ascii()

    +-------------+    
    | PromptInput |    
    +-------------+    
           *           
           *           
           *           
+--------------------+ 
| ChatPromptTemplate | 
+--------------------+ 
           *           
           *           
           *           
    +------------+     
    | ChatOpenAI |     
    +------------+     
           *           
           *           
           *           
  +-----------------+  
  | StrOutputParser |  
  +-----------------+  
           *           
           *           
           *           
    +-----------+      
    | add_thank |      
    +-----------+      
           *           
           *           
           *           
 +------------------+  
 | add_thank_output |  
 +------------------+  


**[RunnableParallel]**

**RunnableParallel은 여러 요소가 병렬 처리되도록 처리합니다.**

In [20]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

runnable = RunnableParallel(
    passed=RunnablePassthrough(),
    modified=lambda x: x["num"] + 1,
)

runnable.invoke({"num": 1})

{'passed': {'num': 1}, 'modified': 2}

In [21]:
runnable = RunnableParallel(
    passed=RunnablePassthrough(),
    modified=add_thank,
)

In [22]:
runnable.invoke("안녕하세요")

{'passed': '안녕하세요', 'modified': '안녕하세요 들어주셔서 감사합니다 :)'}

In [24]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

model = ChatOpenAI(model = 'gpt-4o-mini', max_tokens = 128, temperature = 0)

history_prompt = ChatPromptTemplate.from_template("{topic}가 무엇의 약자인지 알려주세요.")
celeb_prompt = ChatPromptTemplate.from_template("{topic} 분야의 유명인사 3명의 이름만 알려주세요.")

output_parser = StrOutputParser()

history_chain = history_prompt | model | output_parser
celeb_chain = celeb_prompt | model | output_parser

map_chain = RunnableParallel(history=history_chain, celeb=celeb_chain)

result = map_chain.invoke({"topic": "AI"})

In [25]:
result

{'history': 'AI는 "Artificial Intelligence"의 약자로, 한국어로는 "인공지능"이라고 합니다. 인공지능은 컴퓨터 시스템이 인간의 지능을 모방하여 학습, 추론, 문제 해결 등의 작업을 수행할 수 있도록 하는 기술을 의미합니다.',
 'celeb': '1. 앤드류 응 (Andrew Ng)\n2. 제프리 힌튼 (Geoffrey Hinton)\n3. 얀 르쿤 (Yann LeCun)'}